[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage4_rag_explain/Stage4_RAG_Explain.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage4_rag_explain/Stage4_RAG_Explain.ipynb

# 📚 Stage 4 — RAG Explain
**CodePilot AI Studio | Module 4 | Homework**

---

## What You Will Learn
- What **RAG** (Retrieval Augmented Generation) is
- What **vector embeddings** are — how text becomes numbers encoding meaning
- How **ChromaDB** stores and searches knowledge by meaning not keywords
- How to build a **persistent knowledge base**
- How the agent **retrieves context** before answering

## The Big Idea
```
WITHOUT RAG:                    WITH RAG:
User asks about code            ┌──────────────────┐
        ↓                       │  Knowledge Base  │
Llama 3 answers from            │  (ChromaDB)      │
training data only              └────────┬─────────┘
(may miss details)                       │ retrieve
                                User asks → find relevant chunks
                                         ↓
                                Llama 3 + context → better answer!
```

> **Analogy:** A student who reads the right textbook chapter BEFORE answering
> the exam — not one who answers from memory alone.

⚠️ **Note:** Stage 4 setup takes about 2 minutes the first time
(installing ChromaDB + downloading embeddings model ~90MB).

⏱ **Expected time: 25 minutes**

## Step 1 — Add Your Groq API Key (One Time Setup)

### Option A — Using Colab Secrets (Recommended!)
Store your key ONCE in Colab Secrets and it works in ALL notebooks automatically:
1. Click the **🔑 key icon** in the left sidebar (or go to Tools → Secrets)
2. Click **'Add new secret'**
3. Name: `GROQ_API_KEY`  (must be exactly this name)
4. Value: paste your key (looks like `gsk_xxxx...`)
5. Toggle **'Notebook access'** to ON
6. Come back and run this cell

### Option B — Paste directly (quick one-time use)
If you do not want to use Secrets, just paste your key in the code cell below.

> Get your free key at **https://console.groq.com** → API Keys → Create API Key

In [ ]:
# ── GROQ API KEY SETUP ───────────────────────────────────────
# This cell tries Colab Secrets first (recommended).
# If not found, falls back to manual paste below.

import os

# ── METHOD 1: Colab Secrets (store once, works in all notebooks)
# If you added GROQ_API_KEY in the Secrets panel, this will find it.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("Groq API key loaded from Colab Secrets!")
        print(f"Key starts with: {GROQ_API_KEY[:8]}...")
    else:
        raise ValueError("Key not found in Secrets")
except Exception as e:
    # ── METHOD 2: Manual paste (fallback)
    # If Secrets is not set up, paste your key here:
    GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace if not using Secrets
    if GROQ_API_KEY == "paste-your-groq-key-here":
        print("ERROR: Key not found in Secrets and not pasted manually.")
        print("")
        print("Option A: Add to Colab Secrets:")
        print("  1. Click the key icon in the left sidebar")
        print("  2. Add secret name: GROQ_API_KEY")
        print("  3. Paste your key as the value")
        print("  4. Enable Notebook access and re-run this cell")
        print("")
        print("Option B: Paste your key directly above (replace 'paste-your-groq-key-here')")
        print("Get your free key from: https://console.groq.com")
    else:
        print(f"Groq API key set manually. Starts with: {GROQ_API_KEY[:8]}...")

# Set as environment variable so LangChain reads it automatically
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Final check
if len(GROQ_API_KEY) > 20:
    print("Ready to proceed!")


## Step 2 — Setup (slightly longer for Stage 4)
ChromaDB and sentence-transformers take ~2 minutes to install. Please wait!

In [ ]:
print("Installing packages for Stage 4 (takes ~2 minutes)...")
!pip install -q langchain-groq langchain langchain-community langchain-core langchain-classic langchain-text-splitters
!pip install -q chromadb langchain-chroma
!pip install -q sentence-transformers langchain-huggingface
print("All packages installed!")

## Step 3 — Understand Embeddings

### What is a vector embedding?
An **embedding** converts text into a list of numbers that encode the **meaning**.
Words with similar meanings get similar numbers.

```
'IndexError'  → [0.23, -0.45, 0.87, 0.12, ...]   (384 numbers)
'list index'  → [0.21, -0.41, 0.89, 0.14, ...]   (similar!)
'banana'      → [0.65,  0.32, -0.12, 0.88, ...]  (very different)
```

This means ChromaDB can find 'IndexError docs' even if you search for 'list out of bounds'!
**Semantic search** — search by meaning, not by exact words.

In [ ]:
# ── CONCEPT DEMO: See embeddings in action ────────────────────
# Watch how similar text gets similar numbers
import os
# Suppress tokenizer warnings in Colab
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Loading embedding model (downloads ~90MB first time)...")
from langchain_huggingface import HuggingFaceEmbeddings

# all-MiniLM-L6-v2: small, fast, good quality — perfect for classroom
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded!")
print()

# Convert three different texts to vectors
v1 = embeddings.embed_query("IndexError in Python")
v2 = embeddings.embed_query("list index out of bounds")   # similar meaning!
v3 = embeddings.embed_query("banana fruit recipe")        # very different

# Calculate similarity (dot product of normalized vectors)
import numpy as np
sim_12 = float(np.dot(v1, v2))   # should be HIGH (similar meaning)
sim_13 = float(np.dot(v1, v3))   # should be LOW (different meaning)

print(f"Vector size: {len(v1)} numbers per text")
print(f"First 5 values of 'IndexError': {[round(x,3) for x in v1[:5]]}")
print()
print("Similarity scores (higher = more similar meaning):")
print(f"  'IndexError' vs 'list index out of bounds' : {sim_12:.3f} HIGH")
print(f"  'IndexError' vs 'banana fruit recipe'      : {sim_13:.3f} LOW")
print()
print("KEY: ChromaDB uses these scores to find relevant documents!")

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 4: RAG Explain
# ============================================================
# Concept  : Long-term memory using RAG + ChromaDB
# New here : vector store, retriever, RAG pipeline
# ============================================================

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── STEP 1: Connect to LLM ────────────────────────────────────
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

# ── STEP 2: Build the knowledge base ──────────────────────────
# These are the Python concepts our agent will retrieve from.
# In production this could be thousands of documents!
PYTHON_KNOWLEDGE = [
    "IndexError in Python: An IndexError occurs when you try to access "
    "a list or tuple using an index that does not exist. "
    "Fix: always check len() before indexing, or use try/except IndexError.",

    "ZeroDivisionError in Python: Raised when you divide by zero. "
    "Fix: always validate the denominator is not zero before dividing. "
    "Pattern: if divisor != 0: result = numerator / divisor",

    "TypeError in Python: Raised when an operation is applied to an object "
    "of the wrong type. Common: mixing strings and numbers. "
    "Fix: use isinstance() to check types, or convert with int(), str(), float().",

    "Python Functions Best Practice: Functions should do ONE thing. "
    "Always add a docstring. Handle edge cases explicitly. "
    "Use type hints for clarity. Return consistent types.",

    "Python Lists: Lists are ordered, mutable, and allow duplicates. "
    "Check if empty with 'if my_list:' not 'if len(my_list) > 0'. "
    "Common operations: append(), remove(), pop(), len(), sorted().",

    "Exception Handling in Python: Use specific except clauses, not bare except. "
    "Use try/except/finally for cleanup. Always log errors. "
    "Pattern: try: ... except ValueError as e: print(f'Error: {e}')",
]

# Convert to Document objects (LangChain standard format)
docs = [Document(page_content=text) for text in PYTHON_KNOWLEDGE]

# Split into smaller chunks for better retrieval
# chunk_size=300 → each chunk is up to 300 characters
# chunk_overlap=30 → chunks share 30 characters to avoid cutting mid-sentence
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

print(f"Knowledge base: {len(PYTHON_KNOWLEDGE)} entries → {len(chunks)} chunks after splitting")
print()

# ── STEP 3: Store in ChromaDB ─────────────────────────────────
# Chroma.from_documents() embeds each chunk and stores in a vector database
# In Colab we use in-memory storage (no persist_directory)
print("Building vector database (embedding all chunks)...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="codepilot_knowledge"
)
print("Vector database ready!")
print()

# ── STEP 4: Create the retriever ──────────────────────────────
# The retriever searches the vector store for relevant chunks
# k=2 means: return the 2 most similar chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}   # retrieve top 2 most relevant chunks
)

# ── STEP 5: The RAG explain function ──────────────────────────
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "code"],
    template="""You are a Python tutor explaining code to a beginner.
Use the knowledge below to give a more informed explanation.

Relevant Knowledge:
{context}

Explain this code:
```python
{code}
```

Structure:
1. WHAT IT DOES: [one sentence]
2. HOW IT WORKS: [step by step]
3. POTENTIAL ISSUES: [any bugs or risks, informed by the knowledge above]
4. BEGINNER TIP: [one practical advice]"""
)

def rag_explain(code: str) -> str:
    """RAG pipeline: Retrieve → Augment → Generate"""
    # Step 1: Retrieve relevant knowledge chunks
    search_query = f"Python concepts and errors relevant to: {code[:100]}"
    relevant_docs = retriever.invoke(search_query)

    # Step 2: Combine chunks into context string
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    print(f"Retrieved {len(relevant_docs)} relevant knowledge chunks:")
    for i, doc in enumerate(relevant_docs):
        print(f"  [{i+1}] {doc.page_content[:80]}...")
    print()

    # Step 3: Augment the prompt with context and generate
    prompt = RAG_PROMPT.format(context=context, code=code)
    response = llm.invoke(prompt)
    return response.content

# ── Test the RAG explain ───────────────────────────────────────
sample_code = """
def get_average(numbers):
    return sum(numbers) / len(numbers)

print(get_average([10, 20, 30]))
print(get_average([]))
"""

print("=" * 60)
print("CodePilot AI Studio — Stage 4: RAG Explain")
print("=" * 60)
print("Code to explain:", sample_code)
print("Searching knowledge base...")
print()

explanation = rag_explain(sample_code)
print("RAG-enhanced explanation:")
print(explanation)
print()
print("KEY: The agent retrieved knowledge about ZeroDivisionError")
print("BEFORE generating the answer — making it more accurate!")

## Step 5 — Verify

In [ ]:
try:
    test_results = retriever.invoke("IndexError list")
    if len(test_results) > 0:
        print("VERIFICATION PASSED")
        print(f"  Knowledge chunks in DB : {len(chunks)}")
        print(f"  Test retrieval         : {len(test_results)} chunks returned")
        print(f"  Sample chunk           : {test_results[0].page_content[:60]}...")
        print("Stage 4 complete! Ready for Stage 5.")
    else:
        print("Retrieval returned empty. Run Step 4 again.")
except:
    print("Run Step 4 first.")

## Step 6 — Experiments

In [ ]:
# ── EXPERIMENT 1: See what gets retrieved ─────────────────────
# Search the knowledge base with different queries.
# Notice how similar-meaning queries return the same chunks!

queries = [
    "IndexError list index out of range",
    "dividing by zero crash",
    "wrong type string number",
]

for q in queries:
    results = retriever.invoke(q)
    print(f"Query: '{q}'")
    print(f"  Retrieved: {results[0].page_content[:80]}...")
    print()

In [ ]:
# ── EXPERIMENT 2: Add your own knowledge ──────────────────────
# Add a new entry and see if it gets retrieved!

new_knowledge = """Python Recursion: A function that calls itself.
Always define a base case to stop recursion.
Without a base case: RecursionError: maximum recursion depth exceeded.
Example: def factorial(n): return 1 if n <= 1 else n * factorial(n-1)"""

# Add to the vector store
new_doc = Document(page_content=new_knowledge)
vectorstore.add_documents([new_doc])
print("Added recursion knowledge to the database!")
print()

# Test retrieval
results = retriever.invoke("recursive function infinite loop")
print("Retrieving with query: 'recursive function infinite loop'")
for r in results:
    print(f"  → {r.page_content[:100]}...")

In [ ]:
# ── EXPERIMENT 3: Compare RAG vs no-RAG ──────────────────────
# Same code, same question — but one uses knowledge base context.

test_code = """
my_list = [1, 2, 3]
print(my_list[10])
"""

# Without RAG: just ask the LLM directly
plain_response = llm.invoke(f"Explain this Python code:\n{test_code}")

# With RAG: retrieve context first, then ask
rag_response = rag_explain(test_code)

print("=== WITHOUT RAG ===")
print(plain_response.content)
print()
print("=== WITH RAG ===")
print(rag_response)
print()
print("Is the RAG version more specific about IndexError?")

## Summary — Stage 4 Complete!

| Concept | What it means |
|---------|---------------|
| **RAG** | Retrieve relevant knowledge, then generate with context |
| **Embedding** | Text converted to numbers encoding meaning |
| **ChromaDB** | Vector database storing embedded knowledge |
| **Retriever** | Searches vector store for similar chunks |
| **`k=2`** | Return top 2 most similar chunks |

➡️ Continue with `stage5_reflection/Stage5_Reflection.ipynb`